# Attention

## 1. The intuition: why should a token look at other tokens?

Notebook 02 ended with two problems of RNNs: they forget, and they are slow.

Attention fixes both. Every token looks directly at every other token.

> A cute teddy bear is reading.

Who is reading? To understand "reading", the model should look at "bear".

![An RNN passes information step by step, attention looks at every token directly](images/attention_intuition.svg)

- **RNN:** the information from "bear" has to pass through "is" to reach "reading". On a long sentence it passes through many steps and fades.
- **Attention:** "reading" looks at "bear" directly, however far away it is.
- A **weight** says how much to look at each token, itself included ("reading" gives itself 8%). The weights sum to 100%.
- The new representation of "reading" is a mix of the tokens it looked at, mostly "bear".
- Next question: how does the model get these weights? That is queries, keys and values.

## 2. The mathematics: how does attention get its weights?

We keep "A cute teddy bear" and use small sizes. Suppose we have:

- Sequence length $T = 4$
- Embedding dimension $d_{\text{model}} = 4$ (the $d$ of notebooks 01 and 02)
- Query and key dimension $d_k = 2$
- Value dimension $d_v = 2$

#### 2.1 The attention formula

$$\boxed{\operatorname{Attention}(Q, K, V) = \underbrace{\operatorname{softmax}}_{\rule{0pt}{3ex}\;\text{how much attention?}\;} \left( \frac{\overbrace{Q K^\top}^{\text{how relevant?}\rule[-2ex]{0pt}{2ex}}}{\underbrace{\sqrt{d_k}}_{\rule{0pt}{3ex}\;\text{keep scores stable}\;}} \right) \underbrace{V}_{\rule{0pt}{3ex}\;\text{what information?}\;}}$$

<br>

Here $Q$, $K$ and $V$ are made from $X$, through three learned matrices $W_Q$, $W_K$, $W_V$:

| | How it is made | Intuition |
|---|---|---|
| **Embeddings** $X$ | Looked up from the embedding table, as in notebook 01.<br>One row per token, $T \times d_{\text{model}}$ | What each token is, before it plays any role. |
| **Queries** $Q$ | $Q = X W_Q$ | What is each token looking for? |
| **Keys** $K$ | $K = X W_K$ | What can each token be matched on? Its label. |
| **Values** $V$ | $V = X W_V$ | What does each token give, if it is chosen? Its content. |

#### 2.2 Understanding "bear", one step at a time

**Goal: a new vector for "bear", built from the other tokens.**

Each step is one part of the formula:

| Step | Part of the formula | What "bear" gets |
|---|---|---|
| 0 | the input $X$ and the three matrices | the starting numbers |
| 1 | $Q = X W_Q$ | its query |
| 2 | $K = X W_K$ and $V = X W_V$ | a key and a value for every token |
| 3 | $Q K^\top$ | a score for every token |
| 4 | $\big/ \sqrt{d_k}$ | scaled scores |
| 5 | $\operatorname{softmax}$ | attention weights |
| 6 | $\cdot\, V$ | its new vector |

**Step 0: the input** (the $X$ and the three matrices)

| Token | Embedding $x$ (from notebooks 01 and 02) |
|---|---|
| A | $[0.1,\ 0.0,\ 0.3,\ -0.1]$ |
| cute | $[0.2,\ 0.5,\ -0.1,\ 0.4]$ |
| teddy | $[0.7,\ 0.1,\ -0.3,\ 0.8]$ |
| bear | $[-0.1,\ 0.9,\ 0.4,\ 0.2]$ |

- The four rows are $X$, of size $T \times d_{\text{model}} = 4 \times 4$. Tokens are rows, as in notebook 01.
- $W_Q$, $W_K$ and $W_V$ turn the one embedding $x$ into three views: a question, a label and some content. In a real model they are learned, and every token uses the same three. Here they are made up, each $4 \times 2$:

$$W_Q = \begin{bmatrix} 0 & 0 \\ 1 & -1 \\ 1 & -1 \\ 1 & 0 \end{bmatrix}, \quad W_K = \begin{bmatrix} 0 & -1 \\ 1 & 0 \\ -1 & 1 \\ 1 & 1 \end{bmatrix}, \quad W_V = \begin{bmatrix} 0 & 1 \\ 1 & 0 \\ 1 & 0 \\ 0 & 0 \end{bmatrix}$$

**Step 1: what is "bear" looking for?** (the $Q$ part)

![bear's embedding times W_Q gives its query](images/attention_step1_query.svg)

- $q = x_{\text{bear}} W_Q$: the embedding of "bear", times $W_Q$, is its **query**.
- A $1 \times 4$ row times a $4 \times 2$ matrix gives a $1 \times 2$ row: $d_k = 2$ numbers.
- This is "bear"'s row of $Q$. Every token has its own query, but to understand "bear" we only need this one.

**Step 2: what does each token offer?** (the $K$ and $V$ parts)

![X times W_K gives the keys, X times W_V gives the values](images/attention_step2_keys_values.svg)

- Every token, "bear" included, gets a **key** and a **value**: $K = X W_K$ and $V = X W_V$.
- "bear" looks at itself too, so its own key and value are used.
- Row $j$ of $K$ is token $j$'s label. Row $j$ of $V$ is its content.
- $Q$, $K$ and $V$ all come from the same $X$, through different matrices. One token, three different vectors.
- These numbers have no labels. Saying "a key means: I describe a noun" is only to help us picture it. In a real model the numbers are learned.

**Step 3: whose label matches bear's question?** (the $Q K^\top$ part)

![bear's query against every key gives one score per token](images/attention_step3_scores.svg)

- $s = q K^\top$: the query of "bear" against every key. We use $K^\top$ ($2 \times 4$) so the shapes fit: $(1 \times 2)(2 \times 4) = 1 \times 4$.
- One score per token, "bear" itself included.
- A big score is a good match: "teddy" ($2.06$) and "cute" ($1.37$) match best. "A" is negative, a poor match.
- This is "bear"'s row of the full score table $Q K^\top$.

**Step 4: is one score taking over?** (the $/\sqrt{d_k}$ part)

![The four scores before and after dividing by the square root of d_k](images/attention_step4_scale.svg)

- Divide each score by $\sqrt{d_k} = \sqrt{2} \approx 1.41$: $s / \sqrt{d_k} = [-0.52,\ 0.97,\ 1.46,\ 0.10]$.
- Why: a dot product adds up $d_k$ products, so longer vectors give bigger scores (about $\sqrt{d_k}$ times bigger).
- Big scores make softmax, the next step, almost all-or-nothing: one token takes nearly all the attention. Dividing keeps the scores in a safe range, whatever $d_k$ is.
- Here $d_k = 2$, so the effect is small. With $d_k = 64$ it matters a lot.
- Only the size changes. The order of the scores stays the same.

**Step 5: how much should I listen to each?** (the softmax part)

![Softmax turns the four scaled scores into shares that sum to 1](images/attention_step5_softmax.svg)

- Softmax takes $e^{\text{score}}$ of each scaled score, then divides by the total: $\alpha_j = e^{s_j} \big/ \sum_\ell e^{s_\ell}$.
- $e^{x}$ makes every number positive (even $-0.52$ gives $0.595$). Dividing by the total, $8.644$, makes the shares add up to 1.
- A bigger score gets a bigger share: the weights are $[0.069,\ 0.305,\ 0.498,\ 0.128]$ for A, cute, teddy and bear.
- These are the attention weights: "bear"'s row of the full weight table. Every row sums to 1.
- This is why step 4 matters: without the scale, "teddy" would take $0.585$ instead of $0.498$.

**Step 6: what do I take from them?** (the $\cdot V$ part)

![Each value times its weight, added up, gives the new bear](images/attention_step6_mix.svg)

- $z = \sum_j \alpha_j v_j$: each value times its weight, then add them all up.
- "teddy" has the biggest weight ($0.498$), so its value counts the most. "A" barely counts ($0.069$).
- The new "bear" is $z_{\text{bear}} \approx [0.21,\ 0.40]$: a mix of all four values.
- It has $d_v = 2$ numbers, the size of a value.
- This is "bear"'s row of the output. That is attention for one token.

**Back to the formula**

$$\operatorname{Attention}(Q, K, V) = \underbrace{\operatorname{softmax}}_{\rule{0pt}{3ex}\;\text{step 5}\;} \left( \frac{\overbrace{Q K^\top}^{\text{step 3}\rule[-2ex]{0pt}{2ex}}}{\underbrace{\sqrt{d_k}}_{\rule{0pt}{3ex}\;\text{step 4}\;}} \right) \underbrace{V}_{\rule{0pt}{3ex}\;\text{step 6}\;}$$

<br>

$Q$, $K$ and $V$ came from steps 1 and 2. For "bear", the result is $z_{\text{bear}} \approx [0.21,\ 0.40]$.

#### 2.3 All the tokens at once

![Queries, scores, weights and output for all four tokens, with the bear row outlined](images/attention_all_tokens.svg)

- The same four steps, for every row at once: one matrix product replaces four vector products.
- $Q = X W_Q$ gives all four queries. $Q K^\top$ gives all 16 scores. Softmax of each row, after the scale, gives the weights. Weights times $V$ gives a new vector for every token.
- The outlined row is "bear": what we built in steps 1, 3, 5 and 6.
- Row $i$ of the weights shows where token $i$ looks. "A" spreads its attention almost evenly ($0.22, 0.27, 0.29, 0.22$), while "bear" puts half on "teddy".

## 3. Causal masking

- A model that writes text (a decoder) produces one token at a time. Token $i$ may only use tokens $1, \dots, i$, because the later ones do not exist yet.
- Training shows the model the whole sentence at once. Without a rule, "cute" could look at "teddy" and "bear", the very words it is supposed to predict.
- The fix: hide the future tokens inside the attention formula.

#### 3.1 The mask

$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}} + \text{mask}\right) V$$

- The mask is a $T \times T$ table of numbers, added to the scores before softmax.
- It has $0$ on and below the diagonal (token $i$ may look at token $j \le i$) and $-\infty$ above it (the future).
- $e^{-\infty} = 0$, so a hidden token gets weight $0$. The visible tokens share the rest, and each row still sums to 1.

![The scaled scores plus the mask give weights that are zero above the diagonal](images/attention_causal_mask.svg)

- "A" can only look at itself, so its weight on itself is $1$.
- "cute" looks at "A" and "cute": $0.31$ and $0.69$. "teddy" and "bear" get $0$.
- "bear" is the last token, so it sees everyone. The mask changes nothing in its row.
- Because of the zeros, the output of token $i$ never depends on later tokens. Changing "bear" cannot change the outputs of "A", "cute" or "teddy".
- It is still one matrix product for all tokens. Without the mask, every token sees every token, as in section 2.

## 4. Attention in PyTorch

We write the formula in a few lines, then test it. Same "A cute teddy bear" example as above.

In [1]:
import math
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=2, sci_mode=False)

# Same numbers as section 2: one row per token (A, cute, teddy, bear)
X = torch.tensor([[ 0.1, 0.0,  0.3, -0.1],
                  [ 0.2, 0.5, -0.1,  0.4],
                  [ 0.7, 0.1, -0.3,  0.8],
                  [-0.1, 0.9,  0.4,  0.2]])
W_Q = torch.tensor([[0., 0.], [1., -1.], [1., -1.], [1., 0.]])
W_K = torch.tensor([[0., -1.], [1., 0.], [-1., 1.], [1., 1.]])
W_V = torch.tensor([[0., 1.], [1., 0.], [1., 0.], [0., 0.]])


def attention(X, causal=False):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V                  # three views of the same X
    scores = Q @ K.T / math.sqrt(K.shape[-1])            # Q K^T / sqrt(d_k)
    if causal:                                            # hide the future: -inf above the diagonal
        future = torch.triu(torch.ones(len(X), len(X)), diagonal=1).bool()
        scores = scores.masked_fill(future, float("-inf"))
    weights = torch.softmax(scores, dim=-1)               # each row sums to 1
    return weights @ V, weights                           # new vector for every token


Z, A = attention(X)
print("weights (row = who looks, column = who is looked at):")
print(A)
print("\noutput Z, one new vector per token:")
print(Z)
print("\nnew 'bear' (last row):", Z[3])

weights (row = who looks, column = who is looked at):
tensor([[0.22, 0.27, 0.29, 0.22],
        [0.13, 0.30, 0.36, 0.21],
        [0.16, 0.28, 0.29, 0.27],
        [0.07, 0.31, 0.50, 0.13]])

output Z, one new vector per token:
tensor([[0.41, 0.26],
        [0.36, 0.30],
        [0.45, 0.25],
        [0.21, 0.40]])

new 'bear' (last row): tensor([0.21, 0.40])


#### 4.1 Check against PyTorch

PyTorch has attention built in: `F.scaled_dot_product_attention`. If our few lines are right, both give the same numbers.

In [2]:
Q, K, V = X @ W_Q, X @ W_K, X @ W_V

# PyTorch's version (it expects a batch dimension, so we add one with [None])
Z_torch = F.scaled_dot_product_attention(Q[None], K[None], V[None])[0]
Z_torch_causal = F.scaled_dot_product_attention(Q[None], K[None], V[None], is_causal=True)[0]

print("ours:")
print(Z)
print("\nPyTorch:")
print(Z_torch)
print()
print("ours == PyTorch (no mask):  ", torch.allclose(Z, Z_torch, atol=1e-6))
print("ours == PyTorch (with mask):", torch.allclose(attention(X, causal=True)[0], Z_torch_causal, atol=1e-6))

ours:
tensor([[0.41, 0.26],
        [0.36, 0.30],
        [0.45, 0.25],
        [0.21, 0.40]])

PyTorch:
tensor([[0.41, 0.26],
        [0.36, 0.30],
        [0.45, 0.25],
        [0.21, 0.40]])

ours == PyTorch (no mask):   True
ours == PyTorch (with mask): True


#### 4.2 Context changes the output

"bear" keeps the same embedding, but its output depends on the other tokens. We swap "teddy" for another word and look at "bear" again.

In [3]:
X2 = X.clone()
X2[2] = torch.tensor([-0.6, 0.3, 0.5, -0.4])   # a different word in the "teddy" position

Z2, A2 = attention(X2)

print("embedding of 'bear' (same in both):", X[3], X2[3])
print("output of 'bear' next to 'teddy':   ", Z[3])
print("output of 'bear' next to the other word:", Z2[3])
print("\nsame outputs?", torch.allclose(Z[3], Z2[3]))

embedding of 'bear' (same in both): tensor([-0.10,  0.90,  0.40,  0.20]) tensor([-0.10,  0.90,  0.40,  0.20])
output of 'bear' next to 'teddy':    tensor([0.21, 0.40])
output of 'bear' next to the other word: tensor([0.63, 0.07])

same outputs? False


- The embedding is a fixed lookup (notebook 01). The attention output is not: it mixes in the neighbours.
- This is how attention gives the same word a different meaning in a different sentence.

#### 4.3 Word order is ignored

We shuffle the tokens, run attention, and compare with shuffling the outputs.

In [4]:
perm = [3, 1, 0, 2]                          # a new order: bear, cute, A, teddy

Z_shuffled_in, _ = attention(X[perm])        # shuffle first, then attention
print("shuffle, then attention == attention, then shuffle:", torch.allclose(Z_shuffled_in, Z[perm], atol=1e-6))
print("\noutput for the shuffled input:")
print(Z_shuffled_in)
print("\noriginal output, rows in the same new order:")
print(Z[perm])

shuffle, then attention == attention, then shuffle: True

output for the shuffled input:
tensor([[0.21, 0.40],
        [0.36, 0.30],
        [0.41, 0.26],
        [0.45, 0.25]])

original output, rows in the same new order:
tensor([[0.21, 0.40],
        [0.36, 0.30],
        [0.41, 0.26],
        [0.45, 0.25]])


- "bear" gets $[0.21,\ 0.40]$ whether it is the first token or the last one. Shuffling the input only shuffles the rows of the output.
- Attention only compares vectors. It never uses the position, so the order has to be put into the embeddings (notebook 04).

#### 4.4 The mask hides the future

We change the last token ("bear") and see which outputs move.

In [5]:
X3 = X.clone()
X3[3] = torch.tensor([0.5, -0.5, 0.8, -0.2])    # a different last token

for causal in (False, True):
    Za, _ = attention(X, causal)
    Zb, _ = attention(X3, causal)
    same = torch.isclose(Za, Zb, atol=1e-6).all(dim=1)
    print("with mask" if causal else "no mask  ", "| outputs unchanged for A, cute, teddy, bear:", same.tolist())

no mask   | outputs unchanged for A, cute, teddy, bear: [False, False, False, False]
with mask | outputs unchanged for A, cute, teddy, bear: [True, True, True, False]


- Without the mask, changing "bear" changes every output, because everyone looks at "bear".
- With the mask, only the last row changes. A token never sees what comes after it.

#### 4.5 The scale keeps softmax spread out

We take random queries and keys, and let $d_k$ grow. For each $d_k$ we look at the scores and at the largest softmax weight among 4 tokens, with and without the scale.

In [6]:
torch.manual_seed(0)
print("  d_k | score std (no scale) | score std (scaled) | largest weight (no scale) | largest weight (scaled)")
for d_k in [2, 16, 128, 1024]:
    q = torch.randn(2000, 1, d_k)                  # 2000 random queries
    k = torch.randn(2000, 4, d_k)                  # 4 random keys for each
    s = (q * k).sum(-1)                            # raw scores q . k, shape (2000, 4)
    s_scaled = s / math.sqrt(d_k)
    top = torch.softmax(s, -1).max(-1).values.mean()
    top_scaled = torch.softmax(s_scaled, -1).max(-1).values.mean()
    print(f"{d_k:5d} | {s.std():20.2f} | {s_scaled.std():18.2f} | {top:25.2f} | {top_scaled:23.2f}")

  d_k | score std (no scale) | score std (scaled) | largest weight (no scale) | largest weight (scaled)
    2 |                 1.44 |               1.02 |                      0.54 |                    0.48
   16 |                 4.00 |               1.00 |                      0.82 |                    0.51
  128 |                11.41 |               1.01 |                      0.94 |                    0.52
 1024 |                32.19 |               1.01 |                      0.98 |                    0.52


- Without the scale, the scores spread out as $d_k$ grows (std $\approx \sqrt{d_k}$). Softmax then puts almost all the weight on one token.
- With the scale, the std stays near 1 for every $d_k$, and the weights stay spread out.
- A softmax that is nearly one-hot has almost no gradient, so the model cannot learn which tokens to look at.

## 5. Summary

$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

- Each token makes a **query**, a **key** and a **value** from its embedding.
- Query against keys gives scores. Scale and softmax turn them into weights that sum to 1.
- The new token is the weighted mix of all values.
- A mask can hide the future tokens, so a token only looks at the ones before it.

| | RNN | Attention |
|---|---|---|
| How a token gets information from another token | Through every step in between | Looks at it directly |
| Steps between two tokens | Up to $T$ | 1 |
| Computation over the sequence | One step after another | All tokens at once |
| Cost per layer | About $T \cdot d^2$ | About $T^2 \cdot d$ (a $T \times T$ table of scores) |
| Word order | Built in, because it reads left to right | Not used, so it has to be added to the embeddings |

- Attention fixes the two RNN problems from notebook 02: far-away information fades, and the steps cannot run in parallel.
- The price is the $T \times T$ table. Doubling the text makes the table four times bigger.
- Still missing: word order, and several ways of looking at once. Both come in notebook 04.